# 匹配问题

假设要分配 5 个人做 5 项不同工作，每个人做不同工作产生的效益由邻接矩阵

$$
\boldsymbol{W} = (w_{ij})_{5 \times 5} = \begin{bmatrix} 
3 & 5 & 5 & 4 & 1 \\
2 & 2 & 0 & 2 & 2 \\
2 & 4 & 4 & 1 & 0 \\
0 & 2 & 2 & 1 & 0 \\
1 & 2 & 1 & 3 & 3 
\end{bmatrix}
$$

表示，即 $ w_{ij}(i,j = 1,2,\cdots,5) $ 表示第 $ i $ 个人干第 $ j $ 项工作的效益，试求使效益达到最大的分配方案。

**解** 构造赋权图 $ G = (V, E, \widetilde{W}) $，顶点集 $ V = \{v_1, v_2, \cdots, v_{10}\} $，其中 $ v_1, v_2, \cdots, v_5 $ 分别表示 5 个人，$ v_6, v_7, v_8, v_9, v_{10} $ 分别表示 5 项工作，邻接矩阵为

$$
\widetilde{W} = \begin{bmatrix} 
O & W \\
O & O 
\end{bmatrix}_{10 \times 10}
$$

则问题归结为求赋权图 $ G $ 的权最大的完美对集。

In [19]:
import numpy as np
import networkx as nx
from networkx.algorithms.matching import max_weight_matching

a = np.array([[3, 5, 5, 4, 1], [2, 2, 0, 2, 2], [2, 4, 4, 1, 0],
             [0, 2, 2, 1, 0], [1, 2, 1, 3, 3]])
b = np.zeros((10, 10))
b[0:5, 5:10] = a

G = nx.Graph(b)

s0 = max_weight_matching(G)  # 返回值{(人员, 工作)}，集合包元组
# 循环内的sorted对每个元组的person和job排序, 外面的sorted对列表每个数组按数组的第一个元素排序(升序)
s = sorted([sorted(w) for w in s0])  

L1 = [x[0] for x in s]
L1 = np.array(L1) + 1  # 人员编号
L2 = [x[1] for x in s]
L2 = np.array(L2) - 4  # 工作编号

c = a[L1-1, L2-1]  # 提取对应的效益
d = c.sum()  # 总效益

display(s0)
display(s)
print(f"工作分配对应关系为:")
print(f"人员编号: {L1}")
print(f"工作编号: {L2}")
print(f"即:")
for i in range(5):
    print(f"第{L1[i]}个人去做第{L2[i]}项工作")
print(f"总工作效益: {d}")


{(0, 8), (1, 5), (4, 9), (6, 3), (7, 2)}

[[0, 8], [1, 5], [2, 7], [3, 6], [4, 9]]

工作分配对应关系为:
人员编号: [1 2 3 4 5]
工作编号: [4 1 3 2 5]
即:
第1个人去做第4项工作
第2个人去做第1项工作
第3个人去做第3项工作
第4个人去做第2项工作
第5个人去做第5项工作
总工作效益: 15


In [25]:
def hungarian_algorithm(graph, num_left, num_right):
    """
    匈牙利算法求解二分图最大匹配，包含增广轨记录
    
    参数:
        graph: 字典，表示二分图，键为左侧顶点，值为右侧邻接顶点列表
        num_left: 左侧顶点数量
        num_right: 右侧顶点数量
    
    返回:
        max_matching: 最大匹配数
        matching: 字典，表示匹配关系，键为右侧顶点，值为左侧顶点
        augmenting_paths: 所有找到的增广轨列表
    """
    # 初始化匹配，右侧顶点初始都未匹配（值为None）
    matching = {v: None for v in range(num_right)}
    max_matching = 0
    augmenting_paths = []  # 存储所有找到的增广轨
    
    def dfs(u, visited, path):
        """深度优先搜索寻找增广路径，并记录路径"""
        # 遍历左侧顶点u的所有邻接右侧顶点
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                current_step = (u, v)  # 将当前边加入路径
                path.append(current_step)
                
                # 如果右侧顶点v未匹配，或其匹配的左侧顶点可以找到其他路径
                if matching[v] is None or dfs(matching[v], visited, path):
                    matching[v] = u  # 更新匹配关系
                    return True
                path.pop()  # 如果找不到增广轨，移除当前边
        return False
    
    # 对每个左侧顶点尝试寻找增广路径
    for u in range(num_left):
        path = []
        if dfs(u, set(), path):
            max_matching += 1
            augmenting_paths.append(path)
    
    return max_matching, matching, augmenting_paths


# 复杂案例调用
# 确保特定代码只在模块被直接运行时执行，而在被其他模块导入时不执行(可在其他脚本导入该脚本，只使用其中的匈牙利算法函数)
if __name__ == "__main__":
    # 定义一个更复杂的二分图
    # 左侧顶点: 0, 1, 2, 3, 4 (5个顶点)
    # 右侧顶点: 0, 1, 2, 3, 4, 5 (6个顶点)
    graph = {
        0: [1, 2],         # 左侧0连接右侧1、2
        1: [0, 3, 4],      # 左侧1连接右侧0、3、4
        2: [1, 3],      # 左侧2连接右侧1、3、5
        3: [2, 5],         # 左侧3连接右侧2、5
        4: [0, 4],     # 左侧4连接右侧0、4、5
        5: [0, 1]
    }
    
    num_left = 5
    num_right = 6
    
    # 调用匈牙利算法
    max_matching, matching, augmenting_paths = hungarian_algorithm(
        graph, num_left, num_right
    )
    
    # 输出结果
    print(f"最大匹配数: {max_matching}")
    print("\n匹配关系（右侧顶点 -> 左侧顶点）:")
    for right in sorted(matching.keys()):
        left = matching[right]
        if left is not None:
            print(f"    右侧顶点{right} -> 左侧顶点{left}")
    
    # 详细输出每条增广轨及其交替特性
    print("\n找到的增广轨及其特性:")
    for i, path in enumerate(augmenting_paths, 1):
        print(f"\n第{i}条增广轨: {path}")
        print("  路径分析:")
        for step, (u, v) in enumerate(path):
            # 判断当前边是否为匹配边
            is_matching_edge = (matching[v] == u)
            
            # 确定当前顶点的匹配状态
            u_matched = any(left == u for left in matching.values())
            v_matched = (matching[v] is not None)
            
            # 输出详细信息
            print(f"    步骤{step+1}: 左侧顶点{u} -> 右侧顶点{v}")
            print(f"        边类型: {'匹配边' if is_matching_edge else '非匹配边'}")
            print(f"        顶点状态: 左侧{u}{'已匹配' if u_matched else '未匹配'}, 右侧{v}{'已匹配' if v_matched else '未匹配'}")


最大匹配数: 5

匹配关系（右侧顶点 -> 左侧顶点）:
    右侧顶点0 -> 左侧顶点4
    右侧顶点1 -> 左侧顶点2
    右侧顶点2 -> 左侧顶点0
    右侧顶点3 -> 左侧顶点1
    右侧顶点5 -> 左侧顶点3

找到的增广轨及其特性:

第1条增广轨: [(0, 1)]
  路径分析:
    步骤1: 左侧顶点0 -> 右侧顶点1
        边类型: 非匹配边
        顶点状态: 左侧0已匹配, 右侧1已匹配

第2条增广轨: [(1, 0)]
  路径分析:
    步骤1: 左侧顶点1 -> 右侧顶点0
        边类型: 非匹配边
        顶点状态: 左侧1已匹配, 右侧0已匹配

第3条增广轨: [(2, 1), (0, 2)]
  路径分析:
    步骤1: 左侧顶点2 -> 右侧顶点1
        边类型: 匹配边
        顶点状态: 左侧2已匹配, 右侧1已匹配
    步骤2: 左侧顶点0 -> 右侧顶点2
        边类型: 匹配边
        顶点状态: 左侧0已匹配, 右侧2已匹配

第4条增广轨: [(3, 2), (0, 1), (2, 3)]
  路径分析:
    步骤1: 左侧顶点3 -> 右侧顶点2
        边类型: 非匹配边
        顶点状态: 左侧3已匹配, 右侧2已匹配
    步骤2: 左侧顶点0 -> 右侧顶点1
        边类型: 非匹配边
        顶点状态: 左侧0已匹配, 右侧1已匹配
    步骤3: 左侧顶点2 -> 右侧顶点3
        边类型: 非匹配边
        顶点状态: 左侧2已匹配, 右侧3已匹配

第5条增广轨: [(4, 0), (1, 3), (2, 1), (0, 2), (3, 5)]
  路径分析:
    步骤1: 左侧顶点4 -> 右侧顶点0
        边类型: 匹配边
        顶点状态: 左侧4已匹配, 右侧0已匹配
    步骤2: 左侧顶点1 -> 右侧顶点3
        边类型: 匹配边
        顶点状态: 左侧1已匹配, 右侧3已匹配
    步骤3: 左侧顶点2 -> 右侧顶点1
        边类型: 匹配边


In [ ]:
"""
匈牙利算法(Kuhn-Munkres)类
"""

import numpy as np

class KuhnMunkres:
    def __init__(self, n, m, max_weight=True):
        """
        初始化 Kuhn-Munkres 算法
        :param n: 左部顶点数
        :param m: 右部顶点数
        :param max_weight: 是否求最大权匹配（True 为最大权，False 为最小权）
        """
        self.n = n  # 左部顶点数
        self.m = m  # 右部顶点数
        self.max_weight = max_weight  # 是否最大权匹配
        self.weight = np.zeros((n, m))  # 权重矩阵
        self.lx = np.zeros(n)  # 左部顶点标号
        self.ly = np.zeros(m)  # 右部顶点标号
        self.match = np.full(m, -1, dtype=int)  # 右部顶点的匹配对象
        self.slack = np.zeros(m)  # 松弛量，记录右部顶点的最小差值
        self.vis_x = np.zeros(n, dtype=bool)  # 左部顶点是否访问
        self.vis_y = np.zeros(m, dtype=bool)  # 右部顶点是否访问
        self.prev = np.full(m, -1, dtype=int)  # 记录路径，用于回溯增广轨

    def add_edge(self, x, y, w):
        """
        添加边的权重
        :param x: 左部顶点（0-based）
        :param y: 右部顶点（0-based）
        :param w: 边的权重
        """
        if self.max_weight:
            self.weight[x][y] = w
        else:
            self.weight[x][y] = -w  # 最小权转换为最大权（取负）

    def bfs(self, x):
        """
        BFS 寻找增广轨
        :param x: 起始左部顶点
        :return: 是否找到增广轨
        """
        queue = []
        queue.append(x)
        self.vis_x[x] = True

        while True:
            while queue:
                x = queue.pop(0)
                for y in range(self.m):
                    if not self.vis_y[y]:
                        gap = self.lx[x] + self.ly[y] - self.weight[x][y]
                        if gap < 1e-6:  # 视为 0（浮点误差）
                            self.vis_y[y] = True
                            self.prev[y] = x
                            if self.match[y] == -1:
                                return True  # 找到增广轨
                            # 将匹配的左部顶点加入队列
                            queue.append(self.match[y])
                            self.vis_x[self.match[y]] = True
                        else:
                            if gap < self.slack[y]:
                                self.slack[y] = gap
                                self.prev[y] = x
            # 调整标号
            alpha = np.inf
            for y in range(self.m):
                if not self.vis_y[y] and self.slack[y] < alpha:
                    alpha = self.slack[y]
            for i in range(self.n):
                if self.vis_x[i]:
                    self.lx[i] -= alpha
            for j in range(self.m):
                if self.vis_y[j]:
                    self.ly[j] += alpha
                else:
                    self.slack[j] -= alpha
            # 重新寻找增广轨
            for y in range(self.m):
                if not self.vis_y[y] and self.slack[y] < 1e-6:
                    self.vis_y[y] = True
                    if self.match[y] == -1:
                        return True  # 找到增广轨
                    queue.append(self.match[y])
                    self.vis_x[self.match[y]] = True

    def solve(self):
        """
        求解最大权完美匹配
        :return: 匹配结果（左部顶点到右部顶点的映射）、最大权值和
        """
        # 初始化左部顶点标号（最大边权）
        for i in range(self.n):
            self.lx[i] = np.max(self.weight[i])

        # 对每个左部顶点寻找增广轨
        for i in range(self.n):
            self.vis_x = np.zeros(self.n, dtype=bool)
            self.vis_y = np.zeros(self.m, dtype=bool)
            self.slack = np.full(self.m, np.inf)
            self.prev = np.full(self.m, -1, dtype=int)
            if self.bfs(i):
                # 回溯更新匹配
                y = np.argwhere(self.vis_y)[0, 0]
                while y != -1:
                    x = self.prev[y]
                    prev_y = self.match[y]
                    self.match[y] = x
                    y = prev_y

        # 计算总权重
        total = 0
        match_result = np.full(self.n, -1, dtype=int)
        for y in range(self.m):
            x = self.match[y]
            if x != -1:
                match_result[x] = y
                total += self.weight[x][y]

        # 若为最小权，转换回原权重
        if not self.max_weight:
            total = -total

        return match_result, total

In [ ]:
import numpy as np

def kuhn_munkres(cost_matrix):
    """
    Kuhn-Munkres算法（匈牙利算法）求解最小权重指派问题
    
    参数:
        cost_matrix: 成本矩阵，形状为n×n，cost_matrix[i][j]表示第i个工人分配第j个任务的成本
    
    返回:
        assignment: 分配结果，assignment[i] = j表示第i个工人分配第j个任务
        min_cost: 最小总成本
    """
    n = len(cost_matrix)
    m = len(cost_matrix[0])
    assert n <= m, "工人数量不能超过任务数量"
    
    u = [0] * (n + 1)
    v = [0] * (m + 1)
    p = [0] * (m + 1)
    way = [0] * (m + 1)
    
    for i in range(1, n + 1):
        p[0] = i
        minv = [float('inf')] * (m + 1)
        used = [False] * (m + 1)
        j0 = 0
        
        while True:
            used[j0] = True
            i0 = p[j0]
            delta = float('inf')
            j1 = 0
            
            for j in range(1, m + 1):
                if not used[j]:
                    cur = cost_matrix[i0 - 1][j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur
                        way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]
                        j1 = j
            
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            
            j0 = j1
            if p[j0] == 0:
                break
        
        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break
    
    # 构建分配结果
    assignment = [0] * n
    for j in range(1, m + 1):
        if p[j] != 0:
            assignment[p[j] - 1] = j - 1
    
    # 计算最小成本
    min_cost = -v[0]
    
    return assignment, min_cost


# 复杂案例调用
if __name__ == "__main__":
    # 案例：6个工人分配到6个任务的成本矩阵
    # 矩阵值表示工人i完成任务j所需的成本（小时）
    cost_matrix = [
        [12, 7, 9, 7, 9, 8],
        [8, 9, 6, 6, 6, 7],
        [7, 17, 12, 14, 12, 16],
        [15, 14, 6, 6, 6, 10],
        [4, 10, 7, 10, 8, 9],
        [13, 14, 10, 10, 10, 8]
    ]
    
    # 转换为numpy数组以便更好地展示
    cost_matrix_np = np.array(cost_matrix)
    print("成本矩阵（行：工人，列：任务，值：成本）：")
    print(cost_matrix_np)
    print()
    
    # 调用Kuhn-Munkres算法
    assignment, min_cost = kuhn_munkres(cost_matrix)
    
    # 输出结果
    print("最优分配方案：")
    for worker, task in enumerate(assignment):
        print(f"工人{worker} -> 任务{task}，成本：{cost_matrix[worker][task]}")
    
    print(f"\n最小总成本：{min_cost}")
    
    # 验证结果是否正确
    total = sum(cost_matrix[i][j] for i, j in enumerate(assignment))
    print(f"验证总成本：{total}（应与最小总成本相等）")


成本矩阵（行：工人，列：任务，值：成本）：
[[12  7  9  7  9  8]
 [ 8  9  6  6  6  7]
 [ 7 17 12 14 12 16]
 [15 14  6  6  6 10]
 [ 4 10  7 10  8  9]
 [13 14 10 10 10  8]]

最优分配方案：
工人0 -> 任务1，成本：7
工人1 -> 任务4，成本：6
工人2 -> 任务0，成本：7
工人3 -> 任务3，成本：6
工人4 -> 任务2，成本：7
工人5 -> 任务5，成本：8

最小总成本：41
验证总成本：41（应与最小总成本相等）


In [1]:
import numpy as np
from collections import deque

def kuhn_munkres_max_weight_matching(weights):
    """
    使用Kuhn-Munkres算法求解二分图的最大权匹配。
    
    参数:
        weights: 二维列表或numpy数组，表示二分图的权重矩阵。
                 weights[i][j]是左侧节点i与右侧节点j之间的权重。
                 
    返回:
        total_weight: 最大权匹配的总权重
        matches: 匹配列表，每个元素是一个元组(左侧节点索引, 右侧节点索引)
    """
    # 将权重转换为numpy数组以便处理
    graph = np.array(weights)
    n, m = graph.shape
    
    # 扩展图为方阵（N x N），不足部分填充0
    N = max(n, m)
    graph_expanded = np.zeros((N, N))
    graph_expanded[:n, :m] = graph
    
    # 初始化顶标
    lx = np.max(graph_expanded, axis=1)  # 左侧顶标取每行最大值
    ly = np.zeros(N)                     # 右侧顶标初始化为0
    
    # 初始化匹配记录
    match_x = -np.ones(N, dtype=int)     # 左侧节点的匹配右侧节点
    match_y = -np.ones(N, dtype=int)     # 右侧节点的匹配左侧节点
    
    # 为每个左侧节点寻找匹配
    for x in range(N):
        # BFS初始化
        slack = np.full(N, np.inf)       # 松弛量（最小可调整值）
        pre = -np.ones(N, dtype=int)     # 记录增广路径的前驱节点
        visited_x = np.zeros(N, dtype=bool)  # 标记左侧节点是否在交错树中
        visited_y = np.zeros(N, dtype=bool)  # 标记右侧节点是否在交错树中
        queue = deque()
        
        visited_x[x] = True
        queue.append(x)
        found = False
        
        while not found:
            # 遍历队列中的左侧节点
            while queue and not found:
                x0 = queue.popleft()
                for y in range(N):
                    if visited_y[y]:
                        continue
                    # 计算顶标差
                    gap = lx[x0] + ly[y] - graph_expanded[x0][y]
                    
                    # 更新松弛量
                    if gap < slack[y]:
                        slack[y] = gap
                        pre[y] = x0
                    
                    # 如果gap为0，则添加y到交错树
                    if gap == 0:
                        visited_y[y] = True
                        if match_y[y] == -1:
                            # 找到增广路径，回溯更新匹配
                            cur_y = y
                            while cur_y != -1:
                                x_prev = pre[cur_y]
                                next_y = match_x[x_prev]
                                match_x[x_prev] = cur_y
                                match_y[cur_y] = x_prev
                                cur_y = next_y
                            found = True
                            break
                        else:
                            # y已被匹配，将其匹配的左侧节点加入队列
                            x_next = match_y[y]
                            if not visited_x[x_next]:
                                visited_x[x_next] = True
                                queue.append(x_next)
            
            if found:
                break
                
            # 调整顶标
            d = np.inf
            for y in range(N):
                if not visited_y[y] and slack[y] < d:
                    d = slack[y]
            
            if d == np.inf:  # 无法调整，匹配失败
                break
                
            # 更新交错树中的顶标
            for i in range(N):
                if visited_x[i]:
                    lx[i] -= d
            for j in range(N):
                if visited_y[j]:
                    ly[j] += d
                else:
                    slack[j] -= d
            
            # 检查新加入相等子图的边
            for y in range(N):
                if not visited_y[y] and slack[y] == 0:
                    visited_y[y] = True
                    if match_y[y] == -1:
                        # 找到增广路径，回溯更新匹配
                        cur_y = y
                        while cur_y != -1:
                            x_prev = pre[cur_y]
                            next_y = match_x[x_prev]
                            match_x[x_prev] = cur_y
                            match_y[cur_y] = x_prev
                            cur_y = next_y
                        found = True
                        break
                    else:
                        x_next = match_y[y]
                        if not visited_x[x_next]:
                            visited_x[x_next] = True
                            queue.append(x_next)
    
    # 提取原图匹配结果
    total_weight = 0
    matches = []
    for i in range(n):  # 遍历原左侧节点
        j = match_x[i]
        if j < m:       # 确保匹配的右侧节点在原图范围内
            total_weight += graph_expanded[i][j]
            matches.append((i, j))
    
    return total_weight, matches

# 示例用法
if __name__ == "__main__":
    # 示例权重矩阵（左侧3节点，右侧3节点）
    weights = [
        [2, 3, 0],
        [4, 1, 2],
        [0, 2, 1]
    ]
    
    total_weight, matches = kuhn_munkres_max_weight_matching(weights)
    print("最大权匹配总权重:", total_weight)
    print("匹配对:", matches)

最大权匹配总权重: 8.0
匹配对: [(0, 1), (1, 0), (2, 2)]
